# 319 — XGBoost Final Pool 학습 + 성능 검증

**참조 source**: `services/ai/train/model_experiment/xgboost_gyeom_robustness.ipynb` (보겸 phase 노트북, 7-feature scope)

## 핵심 결정 (320 합의 + 319 plan 채택값)

- **Final pool 2 features**: `mouse_jerk_mean` (lv3 AUC 0.81) + `mouse_max_speed_px_per_ms` (lv3 AUC 0.72)
- **CV 풀**: 601 trials (lv2_macro 101 + balabit 500). lv2_human 51 학습 제외
- **Held-out final test**: lv3_kde 50 (정교 매크로 generalization)
- **Evaluation set**: lv2_human 51 (single-user sanity, 보겸 1인)
- **Model**: XGBoost (n_estimators=400 / max_depth=3 / lr=0.03 / subsample=0.9 / colsample_bytree=0.9)
- **scale_pos_weight = 4.95** (human 500 / macro 101) — fixed 전체 풀
- **CV strategy**: `StratifiedGroupKFold(n_splits=5)`
  - balabit: `user_id` 10 groups
  - lv2_macro: singleton-per-trial group (분산 효과)
- **Metrics 3-tier**:
  - Primary: ROC AUC
  - Secondary 5개: `block_FP_rate` / `allow_FN_rate_design` (≤0.40) / `allow_FN_rate_operational` (≤0.001) / `review_rate_design` / `review_rate_operational`
  - Tertiary: F1, accuracy, precision@0.75, recall@0.40

## 산출물 (§8 후)

`services/ai/train/model_joblib/xgboost_gyeom_final_pool/` 하위 4 artifacts:
- `model.joblib`, `meta.json`, `metrics.json`, `input_features.json`

## 평가 caveat

- **m3.4**: lv2_human eval n=51 + 보겸 1인 → 일반 public 일반화 X (§6)
- **m5.2**: 320-B baseline (random split + LogReg) ↔ 본 노트북 (StratifiedGroupKFold + XGB) 직접 비교 X. floor reference만 (§5a)
- **m5.3**: mini-eval n=101 + lv2_human 1인 → variance ↑, reference 지표 (§7)
- **m4.1**: lv2_macro overfit 모니터링 = (5-fold CV macro recall) − (lv3_kde recall@0.40). gap > 0.15 → flag
- **m5.4 / m5.5**: operational threshold 0.001 calibration 후속 ticket push


## 1. 데이터 준비

In [1]:
from __future__ import annotations

import json
import re
from pathlib import Path

import numpy as np
import pandas as pd
import yaml

SEED = 42
MODEL_NAME = "xgboost_gyeom_final_pool"
LABEL_MAPPING = {"human": 0, "macro": 1}

ROOT = Path.cwd()
if ROOT.name != "model_experiment":
    raise Exception(f"작업 디렉토리는 model_experiment 여야 함 (현재: {ROOT.name})")

CONFIG_PATH = Path("../../configs/feature_config.yaml")
DATA_DIR = Path("../../data/behavior")
MODEL_DIR = Path("../model_joblib") / MODEL_NAME

# trial_id ranges (services/ai/train/EDA/trial_loader.py 동일)
LV2_RANGE = (900001, 909999)
BALABIT_RANGE = (910001, 910500)
LV3_BALABIT_KDE_RANGE = (930001, 930050)

print("ROOT:", ROOT)
print("CONFIG_PATH:", CONFIG_PATH)
print("DATA_DIR:", DATA_DIR)
print("MODEL_DIR:", MODEL_DIR)


ROOT: C:\Users\SSAFY\Desktop\ai-macro-detection\services\ai\train\model_experiment
CONFIG_PATH: ..\..\configs\feature_config.yaml
DATA_DIR: ..\..\data\behavior
MODEL_DIR: ..\model_joblib\xgboost_gyeom_final_pool


In [2]:
# yaml 로드 — 319 final pool 2 features
with CONFIG_PATH.open("r", encoding="utf-8") as f:
    feature_config = yaml.safe_load(f)

INPUT_FEATURES = feature_config["input_features_gyeom_final_pool"]
FEATURE_NAME_KO = feature_config.get("names_ko", {})
print(f"INPUT_FEATURES (final pool): {INPUT_FEATURES}")

# trial_id range → trial_group 매핑
def trial_group(trial_id):
    if LV2_RANGE[0] <= trial_id <= LV2_RANGE[1]:
        return "lv2"
    if BALABIT_RANGE[0] <= trial_id <= BALABIT_RANGE[1]:
        return "balabit"
    if LV3_BALABIT_KDE_RANGE[0] <= trial_id <= LV3_BALABIT_KDE_RANGE[1]:
        return "lv3_balabit_kde"
    return None

# Balabit user_id 추출 (StratifiedGroupKFold groups param용)
_BALABIT_USER_RE = re.compile(r"^balabit_(user\d+)_")
def parse_balabit_user(session_id):
    if not session_id:
        return None
    m = _BALABIT_USER_RE.match(session_id)
    return m.group(1) if m else None

def load_all_trials(data_dir):
    rows = []
    counters = {"total": 0, "kept": 0, "no_label": 0, "out_of_range": 0}

    for path in sorted(data_dir.glob("trial_*.json")):
        counters["total"] += 1
        try:
            trial_id = int(path.stem.split("_", 1)[1])
        except (IndexError, ValueError):
            continue

        group = trial_group(trial_id)
        if group is None:
            counters["out_of_range"] += 1
            continue

        payload = json.loads(path.read_text(encoding="utf-8"))
        label = payload.get("label")
        if label not in LABEL_MAPPING:
            counters["no_label"] += 1
            continue

        summary = payload.get("summary") or {}
        metrics = payload.get("metrics") or {}
        row = {
            "trial_id": trial_id,
            "label": label,
            "trial_group": group,
            "session_id": summary.get("session_id"),
            "top_user_id": payload.get("user_id"),
        }
        row.update(metrics)
        rows.append(row)
        counters["kept"] += 1

    df = pd.DataFrame(rows).sort_values("trial_id").reset_index(drop=True)

    # lv2 세분화: lv2_human / lv2_macro
    df.loc[(df["trial_group"] == "lv2") & (df["label"] == "human"), "trial_group"] = "lv2_human"
    df.loc[(df["trial_group"] == "lv2") & (df["label"] == "macro"), "trial_group"] = "lv2_macro"

    # balabit user_id (group_id parameter용)
    df["balabit_user_id"] = df["session_id"].apply(parse_balabit_user)

    print(f"load counters: {counters}")
    return df

df = load_all_trials(DATA_DIR)
display(df["trial_group"].value_counts().rename_axis("trial_group").reset_index(name="count"))
print(f"total trials: {len(df)}")

# 3-pool 분리 (결정 #1 + #3)
cv_pool = df[df["trial_group"].isin(["lv2_macro", "balabit"])].reset_index(drop=True)
holdout_lv3 = df[df["trial_group"] == "lv3_balabit_kde"].reset_index(drop=True)
eval_lv2_human = df[df["trial_group"] == "lv2_human"].reset_index(drop=True)

print(f"\ncv_pool: {len(cv_pool)} (target 601)")
print(f"  lv2_macro: {(cv_pool['trial_group'] == 'lv2_macro').sum()} (target 101)")
print(f"  balabit:   {(cv_pool['trial_group'] == 'balabit').sum()} (target 500)")
print(f"holdout_lv3: {len(holdout_lv3)} (target 50)")
print(f"eval_lv2_human: {len(eval_lv2_human)} (target 51)")

assert len(cv_pool) == 601, f"cv_pool size mismatch: {len(cv_pool)} != 601"
assert len(holdout_lv3) == 50, f"holdout_lv3 size mismatch: {len(holdout_lv3)} != 50"
assert len(eval_lv2_human) == 51, f"eval_lv2_human size mismatch: {len(eval_lv2_human)} != 51"
print("\n모든 풀 카운트 검증 통과")


INPUT_FEATURES (final pool): ['mouse_jerk_mean', 'mouse_max_speed_px_per_ms']


load counters: {'total': 702, 'kept': 702, 'no_label': 0, 'out_of_range': 0}


,trial_group,count
0,balabit,500
1,lv2_macro,101
2,lv2_human,51
3,lv3_balabit_kde,50


total trials: 702

cv_pool: 601 (target 601)
  lv2_macro: 101 (target 101)
  balabit:   500 (target 500)
holdout_lv3: 50 (target 50)
eval_lv2_human: 51 (target 51)

모든 풀 카운트 검증 통과


## 2. 데이터 전처리

In [3]:
present_features = [f for f in INPUT_FEATURES if f in df.columns]
missing_features = [f for f in INPUT_FEATURES if f not in df.columns]
print(f"present: {present_features}")
print(f"missing: {missing_features}")

if missing_features:
    raise ValueError(f"필수 features 누락: {missing_features}")

def make_X_y(pool_df):
    X = pool_df[present_features].apply(pd.to_numeric, errors="coerce")
    y = pool_df["label"].map(LABEL_MAPPING).astype(int).to_numpy()
    return X, y

X_cv, y_cv = make_X_y(cv_pool)
X_lv3, y_lv3 = make_X_y(holdout_lv3)
X_lv2h, y_lv2h = make_X_y(eval_lv2_human)

print(f"\nX_cv shape: {X_cv.shape}, y_cv class counts: {dict(zip(*np.unique(y_cv, return_counts=True)))}")
print(f"X_lv3 shape: {X_lv3.shape}, y_lv3 class counts: {dict(zip(*np.unique(y_lv3, return_counts=True)))}")
print(f"X_lv2h shape: {X_lv2h.shape}, y_lv2h class counts: {dict(zip(*np.unique(y_lv2h, return_counts=True)))}")

# NaN 검증 — memory feedback_missing_data_policy: 즉시 정지·보고
for name, X in [("X_cv", X_cv), ("X_lv3", X_lv3), ("X_lv2h", X_lv2h)]:
    nan_count = int(X.isna().sum().sum())
    if nan_count > 0:
        nan_per_feat = X.isna().sum().to_dict()
        raise ValueError(f"{name} NaN 발견 ({nan_count}개): {nan_per_feat} — 정지·보고 필요")
print("\n모든 풀 NaN 0건 검증 통과")


present: ['mouse_jerk_mean', 'mouse_max_speed_px_per_ms']
missing: []

X_cv shape: (601, 2), y_cv class counts: {0: 500, 1: 101}
X_lv3 shape: (50, 2), y_lv3 class counts: {1: 50}
X_lv2h shape: (51, 2), y_lv2h class counts: {0: 51}

모든 풀 NaN 0건 검증 통과


## 3. 모델 준비

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier

SCALE_POS_WEIGHT = 4.95  # human 500 / macro 101 (#3/#4 합의값, 전체 cv_pool 고정)

def build_model(seed=SEED):
    return Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            (
                "clf",
                XGBClassifier(
                    n_estimators=400,
                    max_depth=3,
                    learning_rate=0.03,
                    subsample=0.9,
                    colsample_bytree=0.9,
                    objective="binary:logistic",
                    eval_metric="logloss",
                    scale_pos_weight=SCALE_POS_WEIGHT,
                    tree_method="hist",
                    random_state=seed,
                    n_jobs=-1,
                ),
            ),
        ]
    )

model = build_model(SEED)
print(f"scale_pos_weight: {SCALE_POS_WEIGHT}")
print(f"INPUT_FEATURES: {INPUT_FEATURES}")
print("hyperparam: n_estimators=400, max_depth=3, lr=0.03, subsample=0.9, colsample_bytree=0.9")
model


scale_pos_weight: 4.95
INPUT_FEATURES: ['mouse_jerk_mean', 'mouse_max_speed_px_per_ms']
hyperparam: n_estimators=400, max_depth=3, lr=0.03, subsample=0.9, colsample_bytree=0.9


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('imputer', ...), ('clf', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"missing_values missing_values: int, float, str, np.nan, None or pandas.NA, default=np.nanThe placeholder for the missing values. All occurrences of`missing_values` will be imputed. For pandas' dataframes withnullable integer dtypes with missing values, `missing_values`can be set to either `np.nan` or `pd.NA`.",nan
,"strategy strategy: str or Callable, default='mean'The imputation strategy.- If ""mean"", then replace missing values using the mean along each column. Can only be used with numeric data.- If ""median"", then replace missing values using the median along each column. Can only be used with numeric data.- If ""most_frequent"", then replace missing using the most frequent value along each column. Can be used with strings or numeric data. If there is more than one such value, only the smallest is returned.- If ""constant"", then replace missing values with fill_value. Can be used with strings or numeric data.- If an instance of Callable, then replace missing values using the scalar statistic returned by running the callable over a dense 1d array containing non-missing values of each column... versionadded:: 0.20 strategy=""constant"" for fixed value imputation... versionadded:: 1.5 strategy=callable for custom value imputation.",'median'
,"fill_value fill_value: str or numerical value, default=NoneWhen strategy == ""constant"", `fill_value` is used to replace alloccurrences of missing_values. For string or object data types,`fill_value` must be a string.If `None`, `fill_value` will be 0 when imputing numericaldata and ""missing_value"" for strings or object data types.",None
,"copy copy: bool, default=TrueIf True, a copy of X will be created. If False, imputation willbe done in-place whenever possible. Note that, in the following cases,a new copy will always be made, even if `copy=False`:- If `X` is not an array of floating values;- If `X` is encoded as a CSR matrix;- If `add_indicator=True`.",True
,"add_indicator add_indicator: bool, default=FalseIf True, a :class:`MissingIndicator` transform will stack onto outputof the imputer's transform. This allows a predictive estimatorto account for missingness despite imputation. If a feat

## 4. 모델 학습 + 5-fold CV (§5a)

601 trials cv_pool에서 `StratifiedGroupKFold(n_splits=5)` 학습 + 평가.

- **Group 정의** (m3.6 hybrid): balabit `user_id` 10 groups + lv2_macro singleton-per-trial 101 (분산 효과)
- **Primary**: ROC AUC
- **Secondary 5**: `block_FP_rate` / `allow_FN_rate_design` (≤0.40) / `allow_FN_rate_operational` (≤0.001) / `review_rate_design` / `review_rate_operational`
- **Tertiary**: F1, accuracy, precision@0.75 (OOF), recall@0.40 (OOF)
- **m5.2 caveat**: 320-B baseline (random split + LogReg 단일 feature) 직접 비교 X. floor reference 0.81


In [5]:
from sklearn.metrics import (
    make_scorer,
    f1_score,
    accuracy_score,
    roc_auc_score,
    precision_score,
    recall_score,
)


# P(macro) >= block_threshold인 human 비율 (peak time FP cost)
def block_fp_rate(y_true, y_proba, block_threshold=0.75):
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    is_human = (y_true == 0)
    if is_human.sum() == 0:
        return float("nan")
    return float(((y_proba >= block_threshold) & is_human).sum() / is_human.sum())


# P(macro) <= allow_threshold인 macro 비율 (scalper 통과 cost)
def allow_fn_rate(y_true, y_proba, allow_threshold):
    y_true = np.asarray(y_true)
    y_proba = np.asarray(y_proba)
    is_macro = (y_true == 1)
    if is_macro.sum() == 0:
        return float("nan")
    return float(((y_proba <= allow_threshold) & is_macro).sum() / is_macro.sum())


# allow_threshold < P(macro) < block_threshold 비율 (DL secondary 부하)
def review_rate(y_true, y_proba, allow_threshold, block_threshold=0.75):
    y_proba = np.asarray(y_proba)
    return float(((y_proba > allow_threshold) & (y_proba < block_threshold)).sum() / len(y_proba))


# 3-tier scoring dict (sklearn 1.8 modern API: response_method='predict_proba')
SCORING = {
    "roc_auc": "roc_auc",
    "f1": "f1",
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "block_FP_rate": make_scorer(
        lambda yt, yp: block_fp_rate(yt, yp, 0.75),
        response_method="predict_proba",
    ),
    "allow_FN_rate_design": make_scorer(
        lambda yt, yp: allow_fn_rate(yt, yp, 0.40),
        response_method="predict_proba",
    ),
    "allow_FN_rate_operational": make_scorer(
        lambda yt, yp: allow_fn_rate(yt, yp, 0.001),
        response_method="predict_proba",
    ),
    "review_rate_design": make_scorer(
        lambda yt, yp: review_rate(yt, yp, 0.40, 0.75),
        response_method="predict_proba",
    ),
    "review_rate_operational": make_scorer(
        lambda yt, yp: review_rate(yt, yp, 0.001, 0.75),
        response_method="predict_proba",
    ),
}

print(f"총 {len(SCORING)}개 metrics:")
print("  primary: roc_auc")
print("  secondary 5: block_FP_rate, allow_FN_rate_(design/operational), review_rate_(design/operational)")
print("  tertiary 4: f1, accuracy, precision, recall")


총 10개 metrics:
  primary: roc_auc
  secondary 5: block_FP_rate, allow_FN_rate_(design/operational), review_rate_(design/operational)
  tertiary 4: f1, accuracy, precision, recall


In [6]:
from sklearn.model_selection import StratifiedGroupKFold

# m3.6 Hybrid groups vector:
# - balabit: groups[i] = balabit_user_id (10 unique groups, 각 50 trials)
# - lv2_macro: groups[i] = f"lv2_macro_{trial_id}" (각 trial unique singleton group)

def make_groups(pool_df):
    groups = []
    for _, row in pool_df.iterrows():
        if row["trial_group"] == "balabit":
            uid = row["balabit_user_id"]
            if not uid:
                raise ValueError(f"balabit trial_id={row['trial_id']} has no user_id")
            groups.append(uid)
        elif row["trial_group"] == "lv2_macro":
            groups.append(f"lv2_macro_{row['trial_id']}")
        else:
            raise ValueError(f"unexpected trial_group {row['trial_group']!r} in cv_pool")
    return np.array(groups)


groups_cv = make_groups(cv_pool)
n_balabit_groups = len(np.unique([g for g in groups_cv if not g.startswith("lv2_macro_")]))
n_lv2_macro_groups = len(np.unique([g for g in groups_cv if g.startswith("lv2_macro_")]))
print(f"groups_cv shape: {groups_cv.shape}")
print(f"unique groups: {len(np.unique(groups_cv))}")
print(f"  balabit (user_id): {n_balabit_groups}")
print(f"  lv2_macro (singleton): {n_lv2_macro_groups}")

assert n_balabit_groups == 10, f"balabit groups {n_balabit_groups} != 10"
assert n_lv2_macro_groups == 101, f"lv2_macro singleton groups {n_lv2_macro_groups} != 101"

# StratifiedGroupKFold split + fold별 카운트 assertion
sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=SEED)
fold_stats = []
for fold_idx, (train_idx, test_idx) in enumerate(sgkf.split(X_cv, y_cv, groups=groups_cv)):
    test_pool = cv_pool.iloc[test_idx]
    n_macro = (test_pool["trial_group"] == "lv2_macro").sum()
    n_balabit = (test_pool["trial_group"] == "balabit").sum()
    n_balabit_users = test_pool[test_pool["trial_group"] == "balabit"]["balabit_user_id"].nunique()
    fold_stats.append({
        "fold": fold_idx,
        "test_size": len(test_idx),
        "test_macro": int(n_macro),
        "test_balabit": int(n_balabit),
        "test_balabit_users": int(n_balabit_users),
    })

display(pd.DataFrame(fold_stats))

# 각 fold ~20 lv2_macro + ~100 balabit (2 user-groups). 허용 범위 넓게.
for stat in fold_stats:
    assert 10 <= stat["test_macro"] <= 30, f"fold {stat['fold']} macro count {stat['test_macro']} out of [10,30]"
    assert 50 <= stat["test_balabit"] <= 150, f"fold {stat['fold']} balabit count {stat['test_balabit']} out of [50,150]"
    assert 1 <= stat["test_balabit_users"] <= 3, f"fold {stat['fold']} balabit_users {stat['test_balabit_users']} unexpected"
print("\nfold별 카운트 assertion 통과")


groups_cv shape: (601,)
unique groups: 111
  balabit (user_id): 10
  lv2_macro (singleton): 101


,fold,test_size,test_macro,test_balabit,test_balabit_users
0,0,120,20,100,2
1,1,120,20,100,2
2,2,120,20,100,2
3,3,120,20,100,2
4,4,121,21,100,2



fold별 카운트 assertion 통과


In [7]:
from sklearn.model_selection import cross_validate
import time

start = time.time()
cv_result = cross_validate(
    build_model(SEED),
    X_cv,
    y_cv,
    cv=list(sgkf.split(X_cv, y_cv, groups=groups_cv)),
    scoring=SCORING,
    n_jobs=-1,
    return_train_score=False,
)
elapsed = time.time() - start
print(f"cross_validate elapsed: {elapsed:.2f}s")

cv_summary = pd.DataFrame([
    {
        "metric": metric.replace("test_", ""),
        "mean": float(values.mean()),
        "std": float(values.std(ddof=1)),
        "min": float(values.min()),
        "max": float(values.max()),
    }
    for metric, values in cv_result.items()
    if metric.startswith("test_")
]).sort_values("metric").reset_index(drop=True)

display(cv_summary)

# AUC 0.81 floor 검증 (m5.2 caveat: floor reference, regression target X)
auc_mean = float(cv_summary[cv_summary["metric"] == "roc_auc"]["mean"].iloc[0])
auc_std = float(cv_summary[cv_summary["metric"] == "roc_auc"]["std"].iloc[0])
print(f"\n=== AUC 0.81 floor check (m5.2 caveat applied) ===")
print(f"5-fold CV ROC AUC: mean={auc_mean:.4f}, std={auc_std:.4f}")
print(f"  vs 320-B floor (jerk_mean lv3 AUC 0.81): {'>=' if auc_mean >= 0.81 else '<'} floor")
print(f"  vs target 0.85: {'>=' if auc_mean >= 0.85 else '<'} target")
print(f"  vs std target 0.05: {'OK' if auc_std < 0.05 else 'FLAG'}")


cross_validate elapsed: 3.54s


,metric,mean,std,min,max
0,accuracy,0.978388,0.033100,0.925000,1.000000
1,allow_FN_rate_design,0.038095,0.085184,0.000000,0.190476
2,allow_FN_rate_operational,0.009524,0.021296,0.000000,0.047619
3,block_FP_rate,0.000000,0.000000,0.000000,0.000000
4,f1,0.940649,0.086799,0.808511,1.000000
5,precision,0.940741,0.132508,0.703704,1.000000
6,recall,0.951905,0.082485,0.809524,1.000000
7,review_rate_design,0.040000,0.084861,0.000000,0.191667
8,review_rate_operational,0.249793,0.134998,0.123967,0.466667
9,roc_auc,0.992902,0.011070,0.974762,1.000000



=== AUC 0.81 floor check (m5.2 caveat applied) ===
5-fold CV ROC AUC: mean=0.9929, std=0.0111
  vs 320-B floor (jerk_mean lv3 AUC 0.81): >= floor
  vs target 0.85: >= target
  vs std target 0.05: OK


In [8]:
from sklearn.model_selection import cross_val_predict

# OOF probabilities for threshold-based tertiary metrics
oof_probas = cross_val_predict(
    build_model(SEED),
    X_cv,
    y_cv,
    cv=list(sgkf.split(X_cv, y_cv, groups=groups_cv)),
    method="predict_proba",
    n_jobs=-1,
)
oof_proba_macro = oof_probas[:, 1]
print(f"OOF probas shape: {oof_probas.shape}")
print(f"P(macro) range: [{oof_proba_macro.min():.4f}, {oof_proba_macro.max():.4f}]")

oof_pred_block = (oof_proba_macro >= 0.75).astype(int)
oof_pred_recall = (oof_proba_macro > 0.40).astype(int)

precision_at_075 = precision_score(y_cv, oof_pred_block, zero_division=0)
recall_at_040 = recall_score(y_cv, oof_pred_recall, zero_division=0)

print(f"\n=== Tertiary OOF threshold metrics ===")
print(f"precision@0.75: {precision_at_075:.4f}")
print(f"recall@0.40 (P(macro) > 0.40): {recall_at_040:.4f}")

# m4.1 overfit gap input — §5b lv3_kde recall@0.40과 비교용
cv_macro_recall_mean = float(cv_summary[cv_summary["metric"] == "recall"]["mean"].iloc[0])
print(f"\n=== m4.1 overfit gap input (5-fold CV) ===")
print(f"5-fold CV macro recall mean: {cv_macro_recall_mean:.4f}")
print(f"  -> §5b lv3_kde recall@0.40과 비교: gap = (CV recall) - (lv3_kde recall@0.40)")


OOF probas shape: (601, 2)
P(macro) range: [0.0000, 0.9984]

=== Tertiary OOF threshold metrics ===
precision@0.75: 1.0000
recall@0.40 (P(macro) > 0.40): 0.9604

=== m4.1 overfit gap input (5-fold CV) ===
5-fold CV macro recall mean: 0.9519
  -> §5b lv3_kde recall@0.40과 비교: gap = (CV recall) - (lv3_kde recall@0.40)


## 5. 평가 (held-out + eval + mini-eval)

학습된 모델을 3 set에 적용:
- **§5b lv3_kde held-out** (50 macro) — 정교 매크로 generalization
- **§6 lv2_human eval** (51 human) — single-user sanity check (m3.4 caveat)
- **§7 mini-eval** (lv2_human 51 + lv3_kde 50 = 101) — 운영 시뮬레이션 (m5.3 caveat)

§5b 직전 cv_pool 601 전체로 `final_model` 학습. §8에서 동일 모델 persist.


In [9]:
# §5 prologue: cv_pool 601 전체 학습 -> final_model
final_model = build_model(SEED)
final_model.fit(X_cv, y_cv)
print(f"final_model fitted on cv_pool (n={len(X_cv)})")
print(f"  human: {(y_cv == 0).sum()}, macro: {(y_cv == 1).sum()}")


final_model fitted on cv_pool (n=601)
  human: 500, macro: 101


### §5b. lv3_kde held-out test (50 macro only)

정교 매크로 generalization 명시 평가.

- AUC 정의 X (단일 class — macro only)
- 산출: `allow_FN_rate_design/operational`, `review_rate_design/operational`, `recall@0.40`
- m4.1 overfit gap 산출: `gap = (CV macro recall) - (lv3_kde recall@0.40)`


In [10]:
# lv3_kde held-out 평가
lv3_proba_macro = final_model.predict_proba(X_lv3)[:, 1]
print(f"lv3_kde P(macro): min={lv3_proba_macro.min():.4f}, max={lv3_proba_macro.max():.4f}, mean={lv3_proba_macro.mean():.4f}")

lv3_metrics = {
    "n": len(X_lv3),
    "allow_FN_rate_design": allow_fn_rate(y_lv3, lv3_proba_macro, 0.40),
    "allow_FN_rate_operational": allow_fn_rate(y_lv3, lv3_proba_macro, 0.001),
    "review_rate_design": review_rate(y_lv3, lv3_proba_macro, 0.40, 0.75),
    "review_rate_operational": review_rate(y_lv3, lv3_proba_macro, 0.001, 0.75),
    "recall_at_040": float((lv3_proba_macro > 0.40).sum() / len(lv3_proba_macro)),
}

print(f"\n=== §5b lv3_kde held-out metrics (n={lv3_metrics['n']}) ===")
for k, v in lv3_metrics.items():
    if k != "n":
        print(f"  {k}: {v:.4f}")

print(f"\n=== §5b verification (plan §Verification) ===")
print(f"  allow_FN_rate_operational <= 0.10: {'OK' if lv3_metrics['allow_FN_rate_operational'] <= 0.10 else 'FLAG'} ({lv3_metrics['allow_FN_rate_operational']:.4f})")
print(f"  recall@0.40 >= 0.85: {'OK' if lv3_metrics['recall_at_040'] >= 0.85 else 'FLAG'} ({lv3_metrics['recall_at_040']:.4f})")

# m4.1 overfit gap
overfit_gap = cv_macro_recall_mean - lv3_metrics["recall_at_040"]
print(f"\n=== m4.1 overfit gap ===")
print(f"  5-fold CV macro recall mean: {cv_macro_recall_mean:.4f}")
print(f"  lv3_kde recall@0.40: {lv3_metrics['recall_at_040']:.4f}")
print(f"  gap = CV - lv3_kde = {overfit_gap:+.4f}")
print(f"  gap > 0.15 flag: {'FLAG (lv2_macro memorize 의심)' if overfit_gap > 0.15 else 'OK'}")


lv3_kde P(macro): min=0.0000, max=0.0542, mean=0.0015

=== §5b lv3_kde held-out metrics (n=50) ===
  allow_FN_rate_design: 1.0000
  allow_FN_rate_operational: 0.9200
  review_rate_design: 0.0000
  review_rate_operational: 0.0800
  recall_at_040: 0.0000

=== §5b verification (plan §Verification) ===
  allow_FN_rate_operational <= 0.10: FLAG (0.9200)
  recall@0.40 >= 0.85: FLAG (0.0000)

=== m4.1 overfit gap ===
  5-fold CV macro recall mean: 0.9519
  lv3_kde recall@0.40: 0.0000
  gap = CV - lv3_kde = +0.9519
  gap > 0.15 flag: FLAG (lv2_macro memorize 의심)


### §6. lv2_human eval (51 human only)

single-user generalization sanity check.

**m3.4 caveat**: n=51 + 보겸 1인 (`top.user_id='lv2_001'` × 51) → `block_FP_rate`는 보겸 1인 behavior 반영, 일반 public 일반화 X. **reference 지표만**.

- AUC 정의 X (단일 class — human only)
- 산출: `block_FP_rate` (≥0.75), `review_rate_design/operational`
- precision@0.75: 정의 X (lv2_human all human → ground truth positive 0건)


In [11]:
# lv2_human eval
lv2h_proba_macro = final_model.predict_proba(X_lv2h)[:, 1]
print(f"lv2_human P(macro): min={lv2h_proba_macro.min():.4f}, max={lv2h_proba_macro.max():.4f}, mean={lv2h_proba_macro.mean():.4f}")

lv2h_metrics = {
    "n": len(X_lv2h),
    "block_FP_rate": block_fp_rate(y_lv2h, lv2h_proba_macro, 0.75),
    "review_rate_design": review_rate(y_lv2h, lv2h_proba_macro, 0.40, 0.75),
    "review_rate_operational": review_rate(y_lv2h, lv2h_proba_macro, 0.001, 0.75),
}

print(f"\n=== §6 lv2_human eval metrics (n={lv2h_metrics['n']}, m3.4 caveat) ===")
for k, v in lv2h_metrics.items():
    if k != "n":
        print(f"  {k}: {v:.4f}")

print(f"\n=== §6 verification (plan §Verification) ===")
print(f"  block_FP_rate <= 0.05: {'OK' if lv2h_metrics['block_FP_rate'] <= 0.05 else 'FLAG'} ({lv2h_metrics['block_FP_rate']:.4f})")
print(f"  precision@0.75: undefined (lv2_human all human, ground truth positive 0건)")


lv2_human P(macro): min=0.0000, max=0.0000, mean=0.0000

=== §6 lv2_human eval metrics (n=51, m3.4 caveat) ===
  block_FP_rate: 0.0000
  review_rate_design: 0.0000
  review_rate_operational: 0.0000

=== §6 verification (plan §Verification) ===
  block_FP_rate <= 0.05: OK (0.0000)
  precision@0.75: undefined (lv2_human all human, ground truth positive 0건)


### §7. mini-eval (lv2_human 51 + lv3_kde 50 = 101 결합)

운영 시뮬레이션: 양쪽 학습 미관측 데이터 결합 → AUC + 3-zone + F1 산출 가능.

**m5.3 caveat**: n=101 + lv2_human 1인 → variance ↑. **reference 지표 (gating X)**.


In [12]:
# mini-eval: lv2_human + lv3_kde 결합 (양쪽 학습 미관측)
X_mini = pd.concat([X_lv2h, X_lv3], ignore_index=True)
y_mini = np.concatenate([y_lv2h, y_lv3])
print(f"mini-eval shape: X={X_mini.shape}, y class counts: {dict(zip(*np.unique(y_mini, return_counts=True)))}")

mini_proba_macro = final_model.predict_proba(X_mini)[:, 1]
print(f"mini-eval P(macro): min={mini_proba_macro.min():.4f}, max={mini_proba_macro.max():.4f}")

mini_pred_block = (mini_proba_macro >= 0.75).astype(int)
mini_pred_recall = (mini_proba_macro > 0.40).astype(int)

mini_metrics = {
    "n": len(X_mini),
    "roc_auc": float(roc_auc_score(y_mini, mini_proba_macro)),
    "f1": float(f1_score(y_mini, mini_pred_block)),
    "accuracy": float(accuracy_score(y_mini, mini_pred_block)),
    "block_FP_rate": block_fp_rate(y_mini, mini_proba_macro, 0.75),
    "allow_FN_rate_design": allow_fn_rate(y_mini, mini_proba_macro, 0.40),
    "allow_FN_rate_operational": allow_fn_rate(y_mini, mini_proba_macro, 0.001),
    "review_rate_design": review_rate(y_mini, mini_proba_macro, 0.40, 0.75),
    "review_rate_operational": review_rate(y_mini, mini_proba_macro, 0.001, 0.75),
    "precision_at_075": float(precision_score(y_mini, mini_pred_block, zero_division=0)),
    "recall_at_040": float(recall_score(y_mini, mini_pred_recall, zero_division=0)),
}

print(f"\n=== §7 mini-eval metrics (n={mini_metrics['n']}, m5.3 caveat) ===")
for k, v in mini_metrics.items():
    if k != "n":
        print(f"  {k}: {v:.4f}")

print(f"\n=== §7 verification (plan §Verification) ===")
print(f"  AUC >= 0.85: {'OK' if mini_metrics['roc_auc'] >= 0.85 else 'FLAG'} ({mini_metrics['roc_auc']:.4f})")
print(f"  F1 >= 0.85: {'OK' if mini_metrics['f1'] >= 0.85 else 'FLAG'} ({mini_metrics['f1']:.4f})")


mini-eval shape: X=(101, 2), y class counts: {0: 51, 1: 50}
mini-eval P(macro): min=0.0000, max=0.0542

=== §7 mini-eval metrics (n=101, m5.3 caveat) ===
  roc_auc: 0.6200
  f1: 0.0000
  accuracy: 0.5050
  block_FP_rate: 0.0000
  allow_FN_rate_design: 1.0000
  allow_FN_rate_operational: 0.9200
  review_rate_design: 0.0000
  review_rate_operational: 0.0396
  precision_at_075: 0.0000
  recall_at_040: 0.0000

=== §7 verification (plan §Verification) ===
  AUC >= 0.85: FLAG (0.6200)
  F1 >= 0.85: FLAG (0.0000)
